# 01 — Environment Setup

This notebook creates your personal HydraB HOL environment.
Everything is scoped to your user so you can experiment freely.

**What gets created:**
- Database: `HYDRAB_HOL_<your_username>`
- Schemas: BRONZE, SILVER, GOLD, SYNTHETIC
- Warehouse: `HYDRAB_HOL_WH` (XS, auto-suspend 60s)
- Link to shared BRONZE data (Salesforce CRM + Odos Telemetry)

In [ ]:
-- Set context
USE ROLE ACCOUNTADMIN;

-- Create per-user database
SET user_db = 'HYDRAB_HOL_' || CURRENT_USER();
CREATE DATABASE IF NOT EXISTS IDENTIFIER($user_db);
USE DATABASE IDENTIFIER($user_db);

In [ ]:
-- Create schemas
CREATE SCHEMA IF NOT EXISTS BRONZE;
CREATE SCHEMA IF NOT EXISTS SILVER;
CREATE SCHEMA IF NOT EXISTS GOLD;
CREATE SCHEMA IF NOT EXISTS SYNTHETIC;

In [ ]:
-- Create warehouse
CREATE WAREHOUSE IF NOT EXISTS HYDRAB_HOL_WH
  WAREHOUSE_SIZE = 'X-SMALL'
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE
  INITIALLY_SUSPENDED = FALSE;

USE WAREHOUSE HYDRAB_HOL_WH;

In [ ]:
-- Link BRONZE data from inbound share
CREATE OR REPLACE DATABASE HYDRAB_BRONZE_SHARED
  FROM SHARE GXNIMKH.HV05860.HYDRAB_BRONZE_SHARE;

-- Verify: list tables in the shared database
SHOW TABLES IN DATABASE HYDRAB_BRONZE_SHARED;

In [ ]:
-- Create views in your BRONZE schema pointing to shared data
USE SCHEMA IDENTIFIER($user_db || '.BRONZE');

CREATE OR REPLACE VIEW OPPORTUNITY AS
  SELECT * FROM HYDRAB_BRONZE_SHARED.SALESFORCE.OPPORTUNITY;

CREATE OR REPLACE VIEW ASSET AS
  SELECT * FROM HYDRAB_BRONZE_SHARED.SALESFORCE.ASSET;

CREATE OR REPLACE VIEW ASSET_LOCATION_HISTORY AS
  SELECT * FROM HYDRAB_BRONZE_SHARED.SALESFORCE.ASSET_LOCATION_HISTORY__C;

CREATE OR REPLACE VIEW DEFECT_EVENT AS
  SELECT * FROM HYDRAB_BRONZE_SHARED.SALESFORCE.DEFECT_EVENT__C;

CREATE OR REPLACE VIEW DELIVERY_TRACKING AS
  SELECT * FROM HYDRAB_BRONZE_SHARED.SALESFORCE.DELIVERY_TRACKING__C;

CREATE OR REPLACE VIEW ODOS_EVENTS AS
  SELECT * FROM HYDRAB_BRONZE_SHARED.ODOS.EVENTS;

In [ ]:
-- Enable cross-region inference for Cortex AI
ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'ANY_REGION';

In [ ]:
-- Verify setup: count rows in key tables
SELECT 'OPPORTUNITY' as source, COUNT(*) as rows FROM BRONZE.OPPORTUNITY
UNION ALL
SELECT 'ASSET', COUNT(*) FROM BRONZE.ASSET
UNION ALL
SELECT 'ODOS_EVENTS', COUNT(*) FROM BRONZE.ODOS_EVENTS
UNION ALL
SELECT 'DEFECT_EVENT', COUNT(*) FROM BRONZE.DEFECT_EVENT
UNION ALL
SELECT 'DELIVERY_TRACKING', COUNT(*) FROM BRONZE.DELIVERY_TRACKING;

## Setup Complete!

You now have:
- Your own database with BRONZE/SILVER/GOLD schemas
- Views pointing to the shared Salesforce + Odos data
- A dedicated warehouse

**Next:** Open `02_explore_data.ipynb` to explore the data.